# LLM Tool Calling — Agentic Loop with Real Data

LLMs are stateless text predictors — they cannot browse the internet, query a database, or run code on their own. **Tool calling** (also called *function calling*) bridges that gap: you define a set of tools, and the model decides when to invoke them.

## How the loop works

A single tool-calling exchange takes at least **two round trips** to the model:

| Round | What you send | What the model returns |
|-------|--------------|----------------------|
| 1 | User prompt + tool schemas | A *tool call* request (function name + arguments) |
| 2 | Tool result appended to history | Final natural-language answer |

The model never executes the tool itself — it only *requests* a call by name. Your code dispatches the call, captures the result, appends it to the conversation, and calls the model again. This cycle continues until the model returns a plain text response with no pending tool calls.

## What this notebook demonstrates

- Defining a tool as both a **Python function** and a **JSON schema** (the schema tells the model what the tool does and what arguments it expects)
- A reusable `run_tool_loop` helper that handles the dispatch cycle automatically
- A real-data example: `get_stock_price(ticker)` via **yfinance** (`pip install yfinance`)

In [ ]:
import ollama
import yfinance as yf
import json

OLLAMA_MODEL = 'mistral-nemo:12b-instruct-2407-q4_K_M'

def run_tool_loop(prompt, tools, tool_map, model=OLLAMA_MODEL):
    """Run Ollama tool-calling loop until the model stops requesting tools."""
    messages = [{'role': 'user', 'content': prompt}]
    while True:
        resp = ollama.chat(model=model, tools=tools, messages=messages)
        msg  = resp['message']
        messages.append(msg)

        if not msg.get('tool_calls'):
            return msg['content']

        for tc in msg['tool_calls']:
            fn_name = tc['function']['name']
            fn_args = tc['function']['arguments']
            print(f'  [tool call]   {fn_name}({fn_args})')
            result = tool_map[fn_name](**fn_args)
            print(f'  [tool result] {result}')
            messages.append({'role': 'tool', 'content': json.dumps(result)})

### Inside `run_tool_loop`

```
messages = [user prompt]
loop:
    call ollama.chat(model, tools, messages)
    append assistant message to history
    if no tool_calls in response → return the text (done)
    for each tool_call:
        look up the function by name in tool_map
        call the function with the model-supplied arguments
        append {"role": "tool", "content": <result>} to history
```

A few things worth noting:
- The loop is **`while True`** — the model could request multiple tools in sequence before it is satisfied. Each tool result gets appended before the next model call.
- `fn_args` comes directly from the model as a dict. The model infers the argument values from the user's prompt and the schema descriptions — you never parse them yourself.
- The `print` statements in the loop let you watch the agentic exchange happen in real time, which is useful for debugging and for teaching.

---
## Stock Price Tool

Every tool is defined in two parts that serve different audiences:

| Part | Audience | Purpose |
|------|----------|---------|
| Python function | Your code | Actually fetches the data |
| JSON schema | The model | Describes what the tool does and what arguments to pass |

The schema is what gets sent to Ollama. The model reads the `description` fields to decide *whether* to call this tool and *what value* to use for each argument — it never sees the Python code. Precise, natural-language descriptions are therefore important: a vague description leads to wrong or missed tool calls.

### yfinance

`yf.Ticker(symbol).history(period='1d')` returns a DataFrame of OHLCV data for today's session. We take the most recent `Close` value. `stock.info` provides metadata (company name, exchange, currency) that makes the model's answer more informative.

### The prompt

`STOCK_PROMPT` is deliberately phrased using a company name ("Apple") rather than the ticker ("AAPL"). The model must infer the correct ticker symbol from context — demonstrating that the model performs light reasoning before deciding what arguments to pass.

In [ ]:
STOCK_PROMPT = 'What is the current share price of Apple?'

def get_stock_price(ticker: str) -> dict:
    stock = yf.Ticker(ticker.upper())
    hist  = stock.history(period='1d')
    if hist.empty:
        return {'error': f'No data found for ticker: {ticker}'}
    info = stock.info
    return {
        'ticker':    ticker.upper(),
        'name':      info.get('longName', ticker),
        'price_usd': round(float(hist['Close'].iloc[-1]), 2),
        'currency':  info.get('currency', 'USD'),
        'exchange':  info.get('exchange', '')
    }

STOCK_TOOL = {
    'type': 'function',
    'function': {
        'name': 'get_stock_price',
        'description': 'Get the latest closing stock price for a ticker symbol.',
        'parameters': {
            'type': 'object',
            'properties': {
                'ticker': {'type': 'string', 'description': 'Ticker symbol (e.g. AAPL, TSLA, MSFT)'}
            },
            'required': ['ticker']
        }
    }
}

answer = run_tool_loop(STOCK_PROMPT, tools=[STOCK_TOOL], tool_map={'get_stock_price': get_stock_price})
print('\nFinal answer:', answer)